# M9.3 · Uzamsal filtreleme: konvolüsyon, separable, integral görüntü

Komşuluk işlemlerinin temeli: korelasyon ile konvolüsyonun farkını, ayrılabilir çekirdeklerin işlem kazancını ve integral görüntüyle sabit maliyetli kutu filtresini elle hesaplıyor; saf Python'da keyfi boyutlu korelasyon ve konvolüsyonu yazıp `cv2.filter2D` ile birebir karşılaştırıyor, NumPy'da ayrılabilir kutu filtresi, `cv2.integral` ile aynı integral görüntü, dört aramalı kutu filtresi ve Gauss bulanıklaştırmayı kuruyor, PyTorch'ta kenar doldurma modlarını OpenCV'ninkilerle eşleştirip bilinmeyen bir çekirdeği girdi–çıktı çiftinden geri buluyor ve en sonda dört yöntemin hızını çekirdek boyutuna göre ölçüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-3-spatial-filtering.html](https://m-gonul.github.io/pytorch-lab/m9-3-spatial-filtering.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_3_spatial_filtering.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"skimage\", \"scipy\"}))\n    crop = data.camera()[150:190, 200:236].astype(np.float32)\n    rng = np.random.default_rng(0)\n    K = rng.normal(size=(5, 3)).astype(np.float32)\n    K[0, 0] += 2\n\n    def corr():\n        got = np.array(need(ns, \"correlate2d\")(crop.tolist(), K.tolist()), np.float32)\n        close(got, cv2.filter2D(crop, -1, K, borderType=cv2.BORDER_CONSTANT), atol=1e-3)\n    R.test(\"correlate2d = cv2.filter2D (5 × 3 asimetrik çekirdek)\", corr)\n\n    def conv():\n        got = np.array(need(ns, \"convolve2d\")(crop.tolist(), K.tolist()), np.float32)\n        close(got, cv2.filter2D(crop, -1, K[::-1, ::-1].copy(), borderType=cv2.BORDER_CONSTANT), atol=1e-3)\n        imp = [[0.0] * 7 for _ in range(7)]; imp[3][3] = 1.0\n        close(np.array(ns[\"convolve2d\"](imp, K.tolist()))[1:6, 2:5], K, atol=1e-6, what=\"dürtü yanıtı = çekirdek\")\n    R.test(\"convolve2d: çevrilmiş çekirdek, dürtü yanıtı\", conv)\n\n    def box():\n        for s in (3, 11):\n            got = np.array(need(ns, \"box_filter\")(crop.tolist(), s), np.float32)\n            close(got, cv2.boxFilter(crop, -1, (s, s), normalize=True, borderType=cv2.BORDER_CONSTANT), atol=1e-3, what=f\"{s}×{s}\")\n        u8 = np.rint(np.array(ns[\"box_filter\"](crop.tolist(), 3))).astype(np.uint8)\n        ref = cv2.blur(crop.astype(np.uint8), (3, 3), borderType=cv2.BORDER_CONSTANT)\n        ensure(np.abs(u8.astype(int) - ref.astype(int)).max() <= 1, \"uint8'de ±1 (alıştırmadaki ≤3 sınırının altında)\")\n    R.test(\"box_filter = cv2.boxFilter (sıfır doldurma)\", box)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n    f = img.astype(np.float32)\n\n    def bs():\n        for s in (3, 11, 21):\n            close(need(ns, \"box_separable\")(img, s), cv2.boxFilter(f, -1, (s, s), normalize=True, borderType=cv2.BORDER_CONSTANT), atol=1e-2, what=f\"{s}×{s}\")\n    R.test(\"box_separable = cv2.boxFilter (3, 11, 21)\", bs)\n\n    def ii():\n        A = need(ns, \"integral\")(img)\n        ensure(A.shape == (513, 513) and np.array_equal(A, cv2.integral(img)), \"cv2.integral ile birebir\")\n    R.test(\"integral = cv2.integral\", ii)\n\n    def bi():\n        for s in (3, 21):\n            got = need(ns, \"box_integral\")(img, s)\n            close(got, cv2.boxFilter(f, -1, (s, s), normalize=True, borderType=cv2.BORDER_CONSTANT), atol=1e-2, what=f\"{s}×{s}\")\n        u8 = np.rint(ns[\"box_integral\"](img, 3)).astype(np.uint8)\n        ensure(np.abs(u8.astype(int) - cv2.blur(img, (3, 3), borderType=cv2.BORDER_CONSTANT).astype(int)).max() <= 1, \"uint8 ±1\")\n    R.test(\"box_integral (4 arama) = cv2\", bi)\n\n    def gk():\n        for k in (3, 7, 15):\n            close(need(ns, \"gaussian_kernel\")(k), cv2.getGaussianKernel(k, 0).ravel(), atol=1e-7, what=f\"ksize {k}, σ otomatik\")\n        close(ns[\"gaussian_kernel\"](9, 2.5), cv2.getGaussianKernel(9, 2.5).ravel(), atol=1e-7, what=\"σ = 2.5\")\n    R.test(\"gaussian_kernel = cv2.getGaussianKernel\", gk)\n\n    def gb():\n        for k, s in ((7, 0), (15, 3.0)):\n            close(need(ns, \"gaussian_blur\")(img, k, s), cv2.GaussianBlur(f, (k, k), s), atol=1e-2, what=f\"ksize {k}\")\n    R.test(\"gaussian_blur = cv2.GaussianBlur (REFLECT_101)\", gb)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera()[:200, :240].astype(np.float32)\n    rng = np.random.default_rng(1)\n    K = rng.normal(size=(5, 3)).astype(np.float32)\n\n    def borders():\n        for name, bt in ((\"constant\", cv2.BORDER_CONSTANT), (\"replicate\", cv2.BORDER_REPLICATE), (\"reflect101\", cv2.BORDER_REFLECT_101)):\n            got = need(ns, \"filter2d\")(torch.tensor(img), torch.tensor(K), name).numpy()\n            close(got, cv2.filter2D(img, -1, K, borderType=bt), atol=2e-3, what=name)\n    R.test(\"filter2d = cv2.filter2D (üç kenar modu)\", borders)\n\n    def rec():\n        torch.manual_seed(0)\n        x = torch.rand(128, 128)\n        w_true = torch.tensor([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=torch.float32) * 0.5\n        w_true[0, 0] = 0.3\n        y = need(ns, \"filter2d\")(x, w_true)\n        w = need(ns, \"recover_kernel\")(x, y, 3)\n        close(w, w_true, atol=2e-2)\n        return \"bulunan çekirdek:\\n\" + np.array2string(w.numpy(), precision=3)\n    R.test(\"recover_kernel: beyaz gürültü girdisiyle çekirdeği bul\", rec)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"box_cumsum döngüsüz\", lambda: max_loops(src, 0, func=\"box_cumsum\"))\n    g = torch.Generator().manual_seed(0)\n    x = torch.rand(4, 96, 80, generator=g)\n\n    def eq():\n        for k in (1, 3, 11, 31):\n            ref = F.avg_pool2d(x[:, None], k, stride=1, padding=k // 2, count_include_pad=True)[:, 0]\n            close(need(ns, \"box_cumsum\")(x, k), ref, atol=1e-5, what=f\"k = {k}\")\n    R.test(\"box_cumsum = avg_pool2d (count_include_pad)\", eq)\n    if \"box_cumsum\" not in ns:\n        return\n    X = torch.rand(8, 512, 512, generator=g)\n    rows = []\n    for k in (3, 11, 21, 41):\n        w2 = torch.ones(1, 1, k, k) / (k * k)\n        w1 = torch.ones(1, 1, 1, k) / k\n        t_full = bench(lambda: F.conv2d(X[:, None], w2, padding=k // 2), repeat=3)\n        t_sep = bench(lambda: F.conv2d(F.conv2d(X[:, None], w1, padding=(0, k // 2)), w1.transpose(2, 3), padding=(k // 2, 0)), repeat=3)\n        t_cs = bench(lambda: ns[\"box_cumsum\"](X, k), repeat=3)\n        rows.append([str(k), fmt_time(t_full), fmt_time(t_sep), fmt_time(t_cs)])\n    R.table(\"8 × 512 × 512 kutu filtresi\", [\"k\", \"k × k conv2d\", \"separable\", \"cumsum (senin)\"], rows,\n            note=\"Doğrudan konvolüsyon k² ile, separable k ile büyür; cumsum yöntemi k'den bağımsızdır.\")\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"np.pad / copyMakeBorder kullanılmıyor\", lambda: ensure(\"np.pad\" not in src and \"copyMakeBorder\" not in src, \"kendi indekslerinle yaz\"))\n    img = data.camera()[:40, :50]\n    modes = {\"constant\": cv2.BORDER_CONSTANT, \"replicate\": cv2.BORDER_REPLICATE, \"reflect\": cv2.BORDER_REFLECT,\n             \"reflect101\": cv2.BORDER_REFLECT_101, \"wrap\": cv2.BORDER_WRAP}\n    for name, bt in modes.items():\n        def t(name=name, bt=bt):\n            for r in (1, 3, 7):\n                got = need(ns, \"pad\")(img, r, name)\n                ensure(np.array_equal(got, cv2.copyMakeBorder(img, r, r, r, r, bt, value=0)), f\"{name}, r = {r}\")\n        R.test(f\"{name}\", t)\n",
 "x2": "import numpy as np\nimport cv2\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    cases = {\n        \"Gauss 7×7\": cv2.getGaussianKernel(7, 0) @ cv2.getGaussianKernel(7, 0).T,\n        \"kutu 5×3\": np.ones((5, 3)) / 15,\n        \"Sobel x\": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),\n    }\n    for name, k in cases.items():\n        def t(k=k):\n            r = need(ns, \"separate\")(k)\n            ensure(r is not None, \"ayrılabilir olmalı\")\n            close(np.outer(r[0], r[1]), k, atol=1e-9)\n        R.test(f\"{name} ayrılabilir\", t)\n    for name, k in {\"Laplasyen\": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float), \"rastgele\": np.random.default_rng(0).normal(size=(3, 3))}.items():\n        R.test(f\"{name} ayrılamaz\", (lambda k=k: ensure(need(ns, \"separate\")(k) is None, \"None bekleniyordu\")))\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    crop = data.camera()[100:160, 300:380].astype(np.float32)\n\n    def eq():\n        for s in (3, 15, 31):\n            got = np.array(need(ns, \"box_running\")(crop.tolist(), s), np.float32)\n            close(got, cv2.boxFilter(crop, -1, (s, s), normalize=True, borderType=cv2.BORDER_CONSTANT), atol=1e-2, what=f\"{s}×{s}\")\n    R.test(\"box_running = cv2.boxFilter\", eq)\n\n    def cnt():\n        res = {}\n        for s in (3, 31):\n            ns[\"ops\"] = 0\n            need(ns, \"box_running\")(crop.tolist(), s)\n            res[s] = ns[\"ops\"] / crop.size\n        ensure(res[3] > 0, \"ops sayacı artırılmalı (global ops)\")\n        ensure(res[31] < 1.6 * res[3], f\"piksel başına işlem çekirdekle büyümemeli: {res}\")\n        return f\"piksel başına toplama/çıkarma: 3×3 → {res[3]:.2f}, 31×31 → {res[31]:.2f}\"\n    R.test(\"işlem sayısı çekirdek boyutundan bağımsız\", cnt)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_3_spatial_filtering as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Doğrusal uzamsal filtreleme:** $m\times n$ ($m=2a+1$, $n=2b+1$) bir $w$ çekirdeği her pikselin komşuluğunda kaydırılır.

$$
\text{korelasyon: }(w\star f)(x,y)=\sum_{s=-a}^{a}\sum_{t=-b}^{b}w(s,t)\,f(x+s,y+t),\qquad
\text{konvolüsyon: }(w*f)(x,y)=\sum_{s,t}w(s,t)\,f(x-s,y-t).
$$

Konvolüsyon çekirdeği 180° çevirir; birim dürtüye uygulandığında çekirdeği aynen kopyalar (dürtü yanıtı). Simetrik çekirdeklerde ikisi aynıdır. OpenCV'nin `filter2D`'si ve PyTorch'un `conv2d`'si **korelasyondur**.

**Kenar doldurma:** sıfır (`BORDER_CONSTANT`), kopyalama (`REPLICATE`: aaa|abc), yansıtma (`REFLECT`: cba|abc), kenarı tekrarlamadan yansıtma (`REFLECT_101`: cb|abc, OpenCV'nin çoğu filtrede **varsayılanı**), sarmalama (`WRAP`).

**ayrılabilir filtre:** $w=\mathbf c\,\mathbf r^\top$ (rank 1) ise 2-boyutlu filtre, önce satırlara $\mathbf r$, sonra sütunlara $\mathbf c$ uygulamakla aynıdır: piksel başına $mn$ yerine $m+n$ çarpma. Kutu ve Gauss ayrılabilir; Laplasyen değil.

**integral görüntü** (summed-area table): $A(x,y)=\sum_{r\le x,\,c\le y}I(r,c)$. Herhangi bir dikdörtgenin toplamı dört aramayla:

$$
\sum_{r=r_0}^{r_1}\sum_{c=c_0}^{c_1}I=A(r_1,c_1)-A(r_0-1,c_1)-A(r_1,c_0-1)+A(r_0-1,c_0-1).
$$

`cv2.integral` başına bir sıfır satır/sütun ekler: çıktı $(H+1)\times(W+1)$, $-1$ indeksleri gerekmez.

**Alçak geçiren filtreler:** kutu (ortalama) ve Gauss $G(s,t)=K e^{-(s^2+t^2)/2\sigma^2}$. OpenCV `ksize` verip $\sigma=0$ derseniz $\sigma=0.3((k-1)/2-1)+0.8$ kullanır.

## 1 · Elle

Dürtü yanıtı sezgisi: tek bir 1'den oluşan görüntüye konvolüsyon uygularsan çıktıda çekirdeğin kendisini, korelasyon uygularsan 180° dönmüş hâlini görürsün.

Integral görüntünün bit sayısı: $\lceil\log_2(255\cdot2^{20}+1)\rceil=28$. 32 bit yeter; 8 bitlik bir akümülatör ilk birkaç pikselde taşar (ödevin son sorusu).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Görüntü parçası $\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\end{bmatrix}$, merkez pikselde**

- `corr`: Korelasyon, $w=\begin{bmatrix}1&0&0\\0&0&0\\0&0&0\end{bmatrix}$
- `conv`: Konvolüsyon, aynı $w$ (çekirdek 180° döner)
- `lap`: Korelasyon, $w=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$

**Ayrılabilirlik ve işlem sayısı (piksel başına)**

- `mul9`: $\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$ doğrudan: çarpma
- `mul6`: Aynı çekirdek $[1,2,1]^\top[1,2,1]$ olarak ayrılmış: çarpma
- `add5`: 5 × 5 kutu, doğrudan toplam: toplama
- `addi`: 5 × 5 kutu, integral görüntüyle: toplama/çıkarma

**Integral görüntü ve Gauss**

- `ii`: $I=\begin{bmatrix}1&2\\3&4\end{bmatrix}$ için $A(1,1)=\sum_{r\le1,c\le1}I$
- `sig`: OpenCV otomatik $\sigma$, `ksize = 7`: $0.3\,((k-1)/2-1)+0.8$
- `bits`: 1024 × 1024 8 bitlik görüntünün integralini taşmadan tutacak en az bit

In [ ]:
cevaplar = dict(
    corr=None,
    conv=None,
    lap=None,
    mul9=None,
    mul6=None,
    add5=None,
    addi=None,
    ii=None,
    sig=None,
    bits=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"corr":[1,0],"conv":[9,0],"lap":[0,0],"mul9":[9,0],"mul6":[6,0],"add5":[24,0],"addi":[3,0],"ii":[10,0],"sig":[1.4,0],"bits":[28,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; görüntü ve çekirdek liste listesi, çekirdek boyutları tek.

- `correlate2d(img, k)` ve `convolve2d(img, k)` — sıfır doldurma, çıkış girişle aynı boyutta.
- `box_filter(img, size)` — `size × size` ortalama, sıfır doldurma (kenarda da `size²`'ye böl).

Kontrol `cv2.filter2D(…, borderType=cv2.BORDER_CONSTANT)` ve `cv2.boxFilter(…, normalize=True, borderType=cv2.BORDER_CONSTANT)` ile float32'de karşılaştırır. (Alıştırmadaki gibi `cv2.blur`'u varsayılan kenarıyla kullanırsan kenarlar farklı çıkar: varsayılan `REFLECT_101`.)

In [ ]:
%%writefile stage_b.py
def correlate2d(img, k):
    ...


def convolve2d(img, k):
    ...


def box_filter(img, size):
    ...


img = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
w = [[1.0, 0, 0], [0, 0, 0], [0, 0, 0]]
print(correlate2d(img, w)[1][1], convolve2d(img, w)[1][1])   # 1.0 9.0

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok** (çekirdek boyutu üzerinde de).

- `box_separable(img, size)` — sıfır doldurma; önce yatay, sonra dikey 1-boyutlu ortalama. Kayan toplamı birikimli toplamla al: pencere toplamı `cs[i+size] - cs[i]`.
- `integral(img)` → `(H+1, W+1)` `int64`, ilk satır/sütun sıfır (`cv2.integral` gibi).
- `box_integral(img, size)` — görüntüyü `r = size//2` sıfırla doldur, integralini al, her çıkış için **4 arama + 1 bölme**.
- `gaussian_kernel(ksize, sigma=0)` → `(ksize,)`, toplamı 1; `sigma ≤ 0` ise OpenCV kuralı. **İnce ayrıntı:** OpenCV `sigma ≤ 0` ve `ksize ∈ {1, 3, 5, 7}` için formülü değil, binom katsayılarına yakın sabit tabloları kullanır: 3 → $[1,2,1]/4$, 5 → $[1,4,6,4,1]/16$, 7 → $[1,3.5,7,9,7,3.5,1]/32$. Bunları da destekle (`cv2.getGaussianKernel(3, 0)` ile kendin gör).
- `gaussian_blur(img, ksize, sigma=0)` — ayrılabilir, `REFLECT_101` doldurma (`np.pad(…, mode="reflect")`).

In [ ]:
%%writefile stage_c.py
import numpy as np


def box_separable(img, size):
    ...


def integral(img):
    ...


def box_integral(img, size):
    ...


def gaussian_kernel(ksize, sigma=0):
    ...


def gaussian_blur(img, ksize, sigma=0):
    ...


from skimage import data
img = data.camera()
print(integral(img)[-1, -1], box_integral(img, 21).shape, gaussian_kernel(7).round(4))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `filter2d(img, k, border)` — PyTorch'ta korelasyon (`F.conv2d`); `border` OpenCV adıyla: `"constant"`, `"replicate"`, `"reflect101"` → `F.pad` modları `constant`, `replicate`, `reflect`.
- `recover_kernel(x, y, ksize, steps=500, lr=0.1)` — `y = filter2d(x, w*, "reflect101")` olan bilinmeyen `w*`'ı, `w` parametresini `Adam` ile MSE'yi küçülterek bul. Bir CNN katmanının "öğrendiği" şey tam olarak budur.

Deney: aynı şeyi doğal bir görüntüyle (`camera`) dene. Kayıp neredeyse sıfıra iner ama bulunan çekirdek gerçeğinden çok farklıdır: komşu pikseller birbirine çok benzediği için normal denklemler kötü koşullanmıştır, birçok çekirdek neredeyse aynı çıktıyı verir. Sistem tanımada bu yüzden **beyaz gürültü** girdisi kullanılır; kontrol de onu kullanıyor.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def filter2d(img, k, border="reflect101"):
    """img (H, W), k (m, n): korelasyon, çıkış aynı boyut."""
    ...


def recover_kernel(x, y, ksize, steps=500, lr=0.1):
    ...


from skimage import data
x = torch.tensor(data.camera(), dtype=torch.float32)[:128, :128] / 255
print(filter2d(x, torch.ones(3, 3) / 9).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`box_cumsum(x, k)` — `x: (B, H, W)`, sıfır doldurmalı `k × k` ortalama, iki eksende `cumsum` ile (çekirdek boyutundan bağımsız maliyet), **döngü yok**. Kontrol `F.avg_pool2d(…, stride=1, padding=k//2, count_include_pad=True)` ile karşılaştırır ve dört yöntemin süresini $k$'ye göre tablolar.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def box_cumsum(x, k):
    """x (B, H, W); sıfır doldurmalı k × k ortalama. Döngü yok."""
    ...


x = torch.rand(2, 64, 64)
print(box_cumsum(x, 5).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Kenar doldurma modları = cv2.copyMakeBorder

`pad(img, r, mode)` — `img`'yi her kenardan `r` piksel doldur. `mode`: `"constant"` (0), `"replicate"`, `"reflect"` (kenarı tekrar eder), `"reflect101"` (etmez), `"wrap"`. NumPy indeks dizileriyle, `np.pad` kullanmadan: önce `-r … W+r-1` indekslerini hesapla, sonra kurala göre geçerli aralığa eşle.

Kontrol `cv2.copyMakeBorder` ile birebir karşılaştırır.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def pad(img, r, mode):
    ...


a = np.arange(12).reshape(3, 4)
print(pad(a, 2, "reflect101"))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Ayrılabilirlik testi: SVD ile rank-1 ayrıştırma

- `separate(k, tol=1e-8)` → ayrılabilirse `(col, row)` (`np.outer(col, row) ≈ k`), değilse `None`. SVD: $k=U\Sigma V^\top$; $\sigma_2/\sigma_1<$ `tol` ise rank 1, `col = √σ₁ u₁`, `row = √σ₁ v₁`.

Gauss, kutu ve Sobel ayrılabilir; Laplasyen ve rastgele çekirdekler değil.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def separate(k, tol=1e-8):
    ...


import cv2
g = cv2.getGaussianKernel(5, 0) @ cv2.getGaussianKernel(5, 0).T
print(separate(g))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Kayan pencereli kutu filtresi: piksel başına O(1), integral görüntüsüz

Integral görüntü olmadan da piksel başına sabit maliyet mümkün: satır boyunca kayan bir toplam tut; pencere bir sağa kayınca soldan çıkan pikseli çıkar, sağdan gireni ekle.

- `box_running(img, size)` — saf Python, sıfır doldurma, önce satırlarda kayan toplam, sonra sütunlarda. Piksel başına 2 toplama/çıkarma (her eksende), çekirdek boyutundan bağımsız.

Kontrol sonucu `cv2.boxFilter` ile ve **işlem sayısını** bir sayaçla doğrular: `ops` adlı global bir sayaca her toplama/çıkarmada 1 ekle.

In [ ]:
%%writefile extra_x3.py
ops = 0


def box_running(img, size):
    ...


img = [[float(x + y) for x in range(8)] for y in range(6)]
print(box_running(img, 3)[2][2], ops)

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Recursive (IIR) Gauss: çekirdek boyutundan bağımsız bulanıklaştırma

[Young & van Vliet — Recursive implementation of the Gaussian filter (Signal Processing 1995)](https://doi.org/10.1016/0165-1684(95)00020-E)

Gauss bulanıklaştırmayı her eksende ileri-geri üçüncü dereceden bir özyinelemeli (IIR) filtreyle uygula: maliyet $\sigma$'dan bağımsız. Doğrudan (FIR) separable Gauss ve tekrarlı kutu filtresiyle (3 kez kutu ≈ Gauss) doğruluk ve hızı karşılaştır.


**Adımlar**

1. Young–van Vliet katsayılarını ($q$, $b_0..b_3$, $B$) $\sigma$'dan hesapla.
2. Her satırı soldan sağa (nedensel) sonra sağdan sola (anti-nedensel) filtrele; sınır koşullarını tekrarlı başlangıçla kur.
3. Aynısını sütunlara uygula. `cv2.GaussianBlur` (büyük `ksize`) ile hatayı ölç.
4. $\sigma\in\{1,4,16,64\}$ için üç yöntemin süresini ve maksimum hatasını tablola.

**Kabul ölçütleri**

- $\sigma\ge2$ için iç bölgede `cv2.GaussianBlur` ile en büyük fark < 1 gri seviye (8 bit).
- Süre $\sigma=4$ ile $\sigma=64$ arasında %20'den fazla değişmiyor.
- Üç kutu filtresi yaklaşımı için hata ve süre raporlanmış.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 3.4–3.5](https://www.imageprocessingplace.com/) — ©
- [OpenCV — Image Filtering (filter2D, blur, boxFilter, GaussianBlur, integral, copyMakeBorder)](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — Apache-2.0
- [Crow — Summed-area tables for texture mapping (SIGGRAPH 1984)](https://doi.org/10.1145/964965.808600) — ©
- [Szeliski — Computer Vision: Algorithms and Applications, Bölüm 3.2](https://szeliski.org/Book/) — © (PDF ücretsiz)